# Sherweb Support Center — Forecasting Pipeline
Run each cell in order. Each step is self-contained.

## STEP 1 — Load and Explore the Data

In [ ]:
import pandas as pd

df = pd.read_csv('data.csv', encoding='utf-8-sig')

print('=== COLUMN NAMES ===')
print(df.columns.tolist())

print('\n=== FIRST 5 ROWS ===')
display(df.head())

df['DT_Initiate_EST_QH'] = pd.to_datetime(df['DT_Initiate_EST_QH'])

print('\n=== DATE RANGE ===')
print(f'  Min date : {df["DT_Initiate_EST_QH"].min()}')
print(f'  Max date : {df["DT_Initiate_EST_QH"].max()}')

print(f'\n=== TOTAL INTERACTIONS ===')
print(f'  {len(df):,} rows')

print('\n=== MISSING VALUES PER COLUMN ===')
print(df.isnull().sum().to_string())

## STEP 2 — Clean the Data

### 2a — Justification : vérification visuelle de la période Noël / Nouvel An

Avant de supprimer des données, on vérifie visuellement que la période Noël/NY
a bien un comportement anormal par rapport au reste de l'année.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.read_csv('data.csv', encoding='utf-8-sig')
df['DT_Initiate_EST_QH'] = pd.to_datetime(df['DT_Initiate_EST_QH'])

# Volume journalier
daily = df.groupby(df['DT_Initiate_EST_QH'].dt.date).size().reset_index()
daily.columns = ['date', 'volume']
daily['date'] = pd.to_datetime(daily['date'])

def is_non_bau(dt):
    if dt.month == 12 and dt.day >= 24:
        return True
    if dt.month == 1 and dt.day <= 2:
        return True
    return False

daily['is_non_bau'] = daily['date'].apply(is_non_bau)
mean_normal = daily[~daily['is_non_bau']]['volume'].mean()

fig, ax = plt.subplots(figsize=(14, 5))

for _, row in daily[daily['is_non_bau']].iterrows():
    ax.axvspan(row['date'] - pd.Timedelta(hours=12),
               row['date'] + pd.Timedelta(hours=12),
               color='red', alpha=0.15)

ax.plot(daily['date'], daily['volume'], color='steelblue', linewidth=1.2, label='Volume journalier')
ax.axhline(mean_normal, color='orange', linestyle='--', linewidth=1.5,
           label=f'Moyenne normale ({mean_normal:.0f} interactions/jour)')

for _, row in daily[daily['is_non_bau']].iterrows():
    ax.annotate(f"{int(row['volume'])}", xy=(row['date'], row['volume']),
                fontsize=7, color='red', ha='center', va='bottom')

ax.set_title('Volume journalier — mise en évidence de la période Noël / Nouvel An', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel("Nombre d'interactions")
red_patch = mpatches.Patch(color='red', alpha=0.3, label='Période non-BAU (Noël / NY)')
ax.legend(handles=[ax.lines[0], ax.lines[1], red_patch])
plt.tight_layout()
plt.show()

print('--- Volume moyen par période ---')
print(f'  Hors fêtes (BAU)  : {mean_normal:.0f} interactions/jour')
print(f'  Période Noël/NY   : {daily[daily["is_non_bau"]]["volume"].mean():.0f} interactions/jour')
print(f'  Réduction         : {(1 - daily[daily["is_non_bau"]]["volume"].mean() / mean_normal)*100:.0f}%')
print('\n--- Détail jour par jour (période fêtes) ---')
print(daily[daily['is_non_bau']][['date', 'volume']].to_string(index=False))

### 2b — Nettoyage : suppression des lignes vides et de la période non-BAU

La période Noël/NY affiche **-40% de volume** (jusqu'à 6 interactions certains jours
vs 92 en moyenne normale). On l'exclut pour ne pas biaiser le modèle de prévision.
Ces lignes sont conservées dans `df_non_bau` pour analyse séparée si nécessaire.

In [ ]:
import pandas as pd

df = pd.read_csv('data.csv', encoding='utf-8-sig')
original_count = len(df)
print(f'Starting rows: {original_count:,}')

df['DT_Initiate_EST_QH'] = pd.to_datetime(df['DT_Initiate_EST_QH'])

before = len(df)
df = df.dropna(subset=['DT_Initiate_EST_QH', 'ID_Activity'])
removed_empty = before - len(df)
print(f'Rows removed (missing timestamp or activity ID): {removed_empty}')

def is_non_bau(dt):
    if dt.month == 12 and dt.day >= 24:
        return True
    if dt.month == 1 and dt.day <= 2:
        return True
    return False

df['is_non_BAU'] = df['DT_Initiate_EST_QH'].apply(is_non_bau)
non_bau_count = df['is_non_BAU'].sum()
print(f'Rows flagged as non-BAU (Xmas/NY period): {non_bau_count:,}')

df_non_bau = df[df['is_non_BAU']].copy()
df_clean   = df[~df['is_non_BAU']].copy()

print(f'\n--- Summary ---')
print(f'  Original rows        : {original_count:,}')
print(f'  Removed (empty)      : {removed_empty:,}')
print(f'  Removed (non-BAU)    : {non_bau_count:,}')
print(f'  Remaining clean rows : {len(df_clean):,}')
print(f'\nClean date range: {df_clean["DT_Initiate_EST_QH"].min()} → {df_clean["DT_Initiate_EST_QH"].max()}')

df_clean.to_csv('/tmp/sherweb_clean.csv', index=False)
print('\nClean dataset saved to /tmp/sherweb_clean.csv')

## STEP 3 — Volume per 15-Minute Interval

In [ ]:
# Step 3 code will be added here

## STEP 4 — Naive Forecast (Baseline)

In [ ]:
# Step 4 code will be added here

## STEP 5 — EWM (Exponential Weighted Moving Average)

In [ ]:
# Step 5 code will be added here

## STEP 6 — OLS Trend (Long-Term Growth)

In [ ]:
# Step 6 code will be added here

## STEP 7 — Z-Score Anomaly Detection

In [ ]:
# Step 7 code will be added here

## STEP 8 — 56-Day Forecast

In [ ]:
# Step 8 code will be added here

## STEP 9 — Summary for Management

In [ ]:
# Step 9 code will be added here